## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 15: Projektarbeit & Best Practices
# Niveau: Fortgeschrittene
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time

np.random.seed(42)
tf.random.set_seed(42)

print("=== Vollständige ML-Pipeline als Klassen ===\n")

# ═══════════════════════════════════════════════════════════
# PIPELINE KOMPONENTEN
# ═══════════════════════════════════════════════════════════

class DataLoader:
    """Lädt und speichert Rohdaten."""
    def __init__(self, dataset='mnist'):
        self.dataset = dataset
        self.raw_data = None
        self.load_time = None

    def load(self):
        t0 = time.time()
        if self.dataset == 'mnist':
            (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
            self.raw_data = {
                'x_train': x_train, 'y_train': y_train,
                'x_test':  x_test,  'y_test':  y_test
            }
        self.load_time = time.time() - t0
        print(f"[DataLoader] Geladen: {self.dataset} in {self.load_time:.2f}s")
        print(f"  Train: {x_train.shape}, Test: {x_test.shape}")
        return self.raw_data


class Preprocessor:
    """Vorverarbeitung der Rohdaten."""
    def __init__(self, flatten=True, normalize=True, val_fraction=0.1):
        self.flatten       = flatten
        self.normalize     = normalize
        self.val_fraction  = val_fraction
        self.preprocess_time = None

    def fit_transform(self, raw_data):
        t0 = time.time()
        x_train = raw_data['x_train'].copy()
        x_test  = raw_data['x_test'].copy()
        y_train = raw_data['y_train'].copy()
        y_test  = raw_data['y_test'].copy()

        if self.normalize:
            x_train = x_train.astype('float32') / 255.0
            x_test  = x_test.astype('float32') / 255.0

        if self.flatten:
            x_train = x_train.reshape(len(x_train), -1)
            x_test  = x_test.reshape(len(x_test),  -1)

        # Validierungsset abschneiden
        n_val = int(len(x_train) * self.val_fraction)
        x_val   = x_train[-n_val:]
        y_val   = y_train[-n_val:]
        x_train = x_train[:-n_val]
        y_train = y_train[:-n_val]

        self.preprocess_time = time.time() - t0
        processed = {
            'x_train': x_train, 'y_train': y_train,
            'x_val':   x_val,   'y_val':   y_val,
            'x_test':  x_test,  'y_test':  y_test
        }
        print(f"[Preprocessor] Verarbeitet in {self.preprocess_time:.2f}s")
        print(f"  Train={len(x_train)}, Val={len(x_val)}, Test={len(x_test)}")
        return processed


class ModelBuilder:
    """Erstellt das neuronale Netz."""
    def __init__(self, hidden_units=(128, 64), activation='relu',
                  dropout=0.2, n_classes=10):
        self.hidden_units = hidden_units
        self.activation   = activation
        self.dropout      = dropout
        self.n_classes    = n_classes
        self.model        = None
        self.build_time   = None

    def build(self, input_shape):
        t0 = time.time()
        model_layers = [
            layers.Dense(self.hidden_units[0], activation=self.activation,
                          input_shape=(input_shape,))
        ]
        if self.dropout > 0:
            model_layers.append(layers.Dropout(self.dropout))
        for units in self.hidden_units[1:]:
            model_layers.append(layers.Dense(units, activation=self.activation))
        model_layers.append(layers.Dense(self.n_classes, activation='softmax'))

        self.model = keras.Sequential(model_layers, name='pipeline_model')
        self.model.compile(
            optimizer=keras.optimizers.Adam(0.001),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        self.build_time = time.time() - t0
        print(f"[ModelBuilder] Modell erstellt in {self.build_time:.2f}s")
        print(f"  Parameter: {self.model.count_params():,}")
        return self.model


class Trainer:
    """Trainiert das Modell."""
    def __init__(self, epochs=20, batch_size=256, patience=5):
        self.epochs     = epochs
        self.batch_size = batch_size
        self.patience   = patience
        self.history    = None
        self.train_time = None

    def train(self, model, data):
        t0 = time.time()
        callbacks = [
            keras.callbacks.EarlyStopping(
                patience=self.patience, restore_best_weights=True)
        ]
        self.history = model.fit(
            data['x_train'], data['y_train'],
            validation_data=(data['x_val'], data['y_val']),
            epochs=self.epochs, batch_size=self.batch_size,
            callbacks=callbacks, verbose=0
        )
        self.train_time = time.time() - t0
        epochs_run  = len(self.history.history['loss'])
        best_val    = max(self.history.history['val_accuracy'])
        print(f"[Trainer] Training in {self.train_time:.1f}s ({epochs_run} Epochen)")
        print(f"  Beste Val-Acc: {best_val:.4f}")
        return self.history


class Evaluator:
    """Evaluiert Modell-Performance."""
    def __init__(self):
        self.eval_time = None
        self.metrics   = {}

    def evaluate(self, model, data):
        t0 = time.time()
        loss, acc = model.evaluate(data['x_test'], data['y_test'], verbose=0)
        self.eval_time = time.time() - t0
        self.metrics = {'test_loss': loss, 'test_accuracy': acc}
        print(f"[Evaluator] Evaluiert in {self.eval_time:.2f}s")
        print(f"  Test-Loss: {loss:.4f}, Test-Acc: {acc:.4f}")
        return self.metrics


class Deployer:
    """Simuliert Modell-Deployment."""
    def __init__(self, model_path='pipeline_model.keras'):
        self.model_path  = model_path
        self.deploy_time = None

    def deploy(self, model):
        """Speichert Modell und simuliert Deployment."""
        t0 = time.time()
        model.save(self.model_path)
        import os
        size_kb = os.path.getsize(self.model_path) / 1024
        self.deploy_time = time.time() - t0
        print(f"[Deployer] Modell gespeichert: {self.model_path} ({size_kb:.1f}KB)")
        print(f"  Deploy-Zeit: {self.deploy_time:.2f}s")

        # Schnelltest mit 5 Samples
        loaded = keras.models.load_model(self.model_path)
        return loaded


# ═══════════════════════════════════════════════════════════
# PIPELINE AUSFÜHREN
# ═══════════════════════════════════════════════════════════

print("="*60)
print("ML PIPELINE AUSFÜHRUNG")
print("="*60)

pipeline_start = time.time()

# 1. Daten laden
loader = DataLoader(dataset='mnist')
raw_data = loader.load()

# 2. Vorverarbeitung
preprocessor = Preprocessor(flatten=True, normalize=True, val_fraction=0.1)
data = preprocessor.fit_transform(raw_data)

# 3. Modell aufbauen
builder = ModelBuilder(hidden_units=(128, 64), activation='relu', dropout=0.2)
model = builder.build(input_shape=data['x_train'].shape[1])
model.summary()

# 4. Training
trainer = Trainer(epochs=25, batch_size=256, patience=5)
history = trainer.train(model, data)

# 5. Evaluation
evaluator = Evaluator()
metrics = evaluator.evaluate(model, data)

# 6. Deployment
deployer = Deployer('pipeline_model.keras')
loaded_model = deployer.deploy(model)

pipeline_time = time.time() - pipeline_start

# ═══════════════════════════════════════════════════════════
# PIPELINE ZEITMETRIKEN
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("PIPELINE ZEITMETRIKEN")
print(f"{'='*60}")
stage_times = [
    ('DataLoader',    loader.load_time),
    ('Preprocessor',  preprocessor.preprocess_time),
    ('ModelBuilder',  builder.build_time),
    ('Trainer',       trainer.train_time),
    ('Evaluator',     evaluator.eval_time),
    ('Deployer',      deployer.deploy_time),
]
for name, t in stage_times:
    pct = t / pipeline_time * 100
    print(f"  {name:15s}: {t:6.2f}s ({pct:4.1f}%)")
print(f"  {'GESAMT':15s}: {pipeline_time:6.2f}s (100.0%)")

# ═══════════════════════════════════════════════════════════
# VISUALISIERUNG
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('ML-Pipeline: Ergebnisse', fontsize=13)

# Lernkurven
ax = axes[0]
ax.plot(history.history['accuracy'],     label='Train-Acc')
ax.plot(history.history['val_accuracy'], label='Val-Acc')
ax.set_title('Lernkurven'); ax.set_xlabel('Epoche'); ax.set_ylabel('Accuracy')
ax.legend(); ax.grid(True)

# Pipeline Zeitverteilung
ax2 = axes[1]
names_t  = [s[0] for s in stage_times]
times_t  = [s[1] for s in stage_times]
colors_t = plt.cm.Set3(np.linspace(0, 1, len(stage_times)))
ax2.pie(times_t, labels=names_t, colors=colors_t, autopct='%1.1f%%', startangle=90)
ax2.set_title('Pipeline-Zeit Verteilung')

# Metriken
ax3 = axes[2]
ax3.axis('off')
summary = f"""Pipeline Ergebnisse:

Test-Accuracy: {metrics['test_accuracy']:.4f}
Test-Loss:     {metrics['test_loss']:.4f}

Gesamt-Zeit: {pipeline_time:.1f}s
  Training:  {trainer.train_time:.1f}s
  Epochen:   {len(history.history['loss'])}

Modell-Parameter: {model.count_params():,}
Modell-Datei: {deployer.model_path}"""
ax3.text(0.05, 0.95, summary, transform=ax3.transAxes, fontsize=10,
          va='top', fontfamily='monospace',
          bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout()
plt.savefig('F15_1_ml_pipeline.png', dpi=100)
plt.show()


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 15: Projektarbeit & Best Practices
# Niveau: Fortgeschrittene
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
import os
import time
from datetime import datetime

np.random.seed(42)
tf.random.set_seed(42)

print("=== Experiment Tracking System ===\n")

# ══════════════════════════════════════════════════════════
# EXPERIMENT LOGGER KLASSE
# ══════════════════════════════════════════════════════════

class ExperimentLogger:
    """
    Vollständiges Experiment-Tracking-System.

    Zeichnet auf:
    - Modell-Architektur (Schichten, Parameter)
    - Hyperparameter (LR, Batch-Size, etc.)
    - Trainingsmetriken pro Epoche
    - Finale Test-Metriken
    - Metadaten (Zeitstempel, Dauer)
    """

    def __init__(self, log_file='experiments.json'):
        self.log_file    = log_file
        self.experiments = self._load()
        self.current_exp = None

    def _load(self):
        """Lädt bestehende Experimente aus JSON."""
        if os.path.exists(self.log_file):
            with open(self.log_file, 'r') as f:
                return json.load(f)
        return []

    def _save(self):
        """Speichert alle Experimente in JSON."""
        with open(self.log_file, 'w') as f:
            json.dump(self.experiments, f, indent=2)

    def start_experiment(self, name, model, hyperparams):
        """Startet ein neues Experiment."""
        architecture = {
            'name': model.name,
            'n_layers': len(model.layers),
            'n_params': model.count_params(),
            'layer_details': [
                {
                    'name': l.name,
                    'type': type(l).__name__,
                    'params': l.count_params(),
                    'output_shape': str(l.output_shape) if hasattr(l, 'output_shape') else 'N/A'
                }
                for l in model.layers
            ]
        }
        self.current_exp = {
            'id':           len(self.experiments) + 1,
            'name':         name,
            'timestamp':    datetime.now().isoformat(),
            'start_time':   time.time(),
            'architecture': architecture,
            'hyperparams':  hyperparams,
            'epoch_metrics': [],
            'final_metrics': None,
            'status':       'running',
            'duration_s':   None
        }
        print(f"[Logger] Experiment #{self.current_exp['id']} gestartet: {name}")
        return self.current_exp['id']

    def log_epoch(self, epoch, train_loss, train_acc, val_loss, val_acc):
        """Protokolliert Metriken für eine Epoche."""
        if self.current_exp is None:
            raise RuntimeError("Kein aktives Experiment! start_experiment() zuerst aufrufen.")
        self.current_exp['epoch_metrics'].append({
            'epoch':      epoch,
            'train_loss': round(float(train_loss), 6),
            'train_acc':  round(float(train_acc), 6),
            'val_loss':   round(float(val_loss), 6),
            'val_acc':    round(float(val_acc), 6)
        })

    def end_experiment(self, test_loss, test_acc):
        """Schließt ein Experiment ab."""
        if self.current_exp is None:
            raise RuntimeError("Kein aktives Experiment!")
        self.current_exp['final_metrics'] = {
            'test_loss':    round(float(test_loss), 6),
            'test_accuracy': round(float(test_acc), 6),
            'best_val_acc': max(e['val_acc'] for e in self.current_exp['epoch_metrics']),
            'n_epochs':     len(self.current_exp['epoch_metrics'])
        }
        self.current_exp['duration_s'] = round(time.time() - self.current_exp['start_time'], 2)
        self.current_exp['status'] = 'completed'
        self.experiments.append(self.current_exp)
        self._save()
        print(f"[Logger] Experiment #{self.current_exp['id']} abgeschlossen: "
              f"Test-Acc={test_acc:.4f}")
        self.current_exp = None

    def print_leaderboard(self, top_n=5):
        """Zeigt Bestenliste der Experimente."""
        if not self.experiments:
            print("Keine Experimente vorhanden.")
            return

        completed = [e for e in self.experiments if e['status'] == 'completed']
        sorted_exp = sorted(completed,
                            key=lambda x: x['final_metrics']['test_accuracy'],
                            reverse=True)

        print("\n" + "="*80)
        print(f"EXPERIMENT-BESTENLISTE (Top {min(top_n, len(sorted_exp))})")
        print("="*80)
        header = f"{'#':>3} | {'Name':25} | {'Test-Acc':9} | {'Val-Acc':8} | {'Epochen':7} | {'Zeit(s)':7} | HPs"
        print(header)
        print("-"*80)

        for rank, exp in enumerate(sorted_exp[:top_n], 1):
            fm = exp['final_metrics']
            hp_str = ', '.join([f"{k}={v}" for k, v in list(exp['hyperparams'].items())[:3]])
            print(f"{rank:3d} | {exp['name']:25} | {fm['test_accuracy']:9.4f} | "
                  f"{fm['best_val_acc']:8.4f} | {fm['n_epochs']:7d} | "
                  f"{exp['duration_s']:7.1f} | {hp_str}")

        return sorted_exp

    def compare_experiments(self, exp_ids):
        """Vergleicht spezifische Experimente."""
        selected = [e for e in self.experiments if e['id'] in exp_ids]
        return selected

# ══════════════════════════════════════════════════════════
# DATEN LADEN
# ══════════════════════════════════════════════════════════
(x_train_all, y_train_all), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train_all = x_train_all.astype('float32') / 255.0
x_test      = x_test.astype('float32') / 255.0
x_train_all = x_train_all.reshape(-1, 784)
x_test      = x_test.reshape(-1, 784)
x_train     = x_train_all[:5000]
y_train     = y_train_all[:5000]

# ══════════════════════════════════════════════════════════
# MEHRERE EXPERIMENTE DURCHFÜHREN
# ══════════════════════════════════════════════════════════
logger = ExperimentLogger('experiments.json')

experiment_configs = [
    {'name': 'Baseline-64',    'units': [64],       'lr': 0.001,  'dropout': 0.0},
    {'name': 'Mittel-128-64',  'units': [128, 64],  'lr': 0.001,  'dropout': 0.2},
    {'name': 'Groß-256-128',   'units': [256, 128], 'lr': 0.001,  'dropout': 0.3},
    {'name': 'Schnell-LR0.01', 'units': [128, 64],  'lr': 0.01,   'dropout': 0.2},
    {'name': 'Langsam-LR0.0001', 'units': [128, 64], 'lr': 0.0001, 'dropout': 0.2},
]

for cfg in experiment_configs:
    print(f"\n--- Experiment: {cfg['name']} ---")

    # Modell erstellen
    model_layers = []
    for i, u in enumerate(cfg['units']):
        if i == 0:
            model_layers.append(layers.Dense(u, activation='relu', input_shape=(784,)))
        else:
            model_layers.append(layers.Dense(u, activation='relu'))
        if cfg['dropout'] > 0:
            model_layers.append(layers.Dropout(cfg['dropout']))
    model_layers.append(layers.Dense(10, activation='softmax'))

    model = keras.Sequential(model_layers, name=cfg['name'].replace('-', '_').lower())
    model.compile(optimizer=keras.optimizers.Adam(cfg['lr']),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Experiment starten
    hyperparams = {'lr': cfg['lr'], 'units': str(cfg['units']), 'dropout': cfg['dropout']}
    exp_id = logger.start_experiment(cfg['name'], model, hyperparams)

    # Training mit Epoch-Logging
    EPOCHS = 15
    history = model.fit(x_train, y_train, epochs=EPOCHS, batch_size=64,
                         validation_split=0.2, verbose=0)
    for epoch_idx in range(len(history.history['loss'])):
        logger.log_epoch(
            epoch=epoch_idx + 1,
            train_loss=history.history['loss'][epoch_idx],
            train_acc=history.history['accuracy'][epoch_idx],
            val_loss=history.history['val_loss'][epoch_idx],
            val_acc=history.history['val_accuracy'][epoch_idx]
        )

    # Evaluation und Experiment abschließen
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    logger.end_experiment(test_loss, test_acc)

# Leaderboard
leaderboard = logger.print_leaderboard()

# ══════════════════════════════════════════════════════════
# VISUALISIERUNG
# ══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Experiment Tracking: Vergleich', fontsize=14)

completed = [e for e in logger.experiments if e['status'] == 'completed']

# Val-Acc Lernkurven
ax = axes[0, 0]
colors = plt.cm.Set1(np.linspace(0, 1, len(completed)))
for exp, color in zip(completed, colors):
    epochs_list = [m['epoch'] for m in exp['epoch_metrics']]
    val_accs    = [m['val_acc'] for m in exp['epoch_metrics']]
    ax.plot(epochs_list, val_accs, color=color, label=exp['name'], alpha=0.8)
ax.set_title('Val-Accuracy aller Experimente')
ax.set_xlabel('Epoche'); ax.set_ylabel('Val-Accuracy')
ax.legend(fontsize=7); ax.grid(True)

# Test-Accuracy Balkendiagramm
ax2 = axes[0, 1]
names = [e['name'] for e in completed]
test_accs = [e['final_metrics']['test_accuracy'] for e in completed]
sorted_pairs = sorted(zip(test_accs, names), reverse=True)
ax2.barh([p[1] for p in sorted_pairs], [p[0] for p in sorted_pairs],
          color='steelblue', alpha=0.8)
ax2.set_title('Test-Accuracy Leaderboard')
ax2.set_xlabel('Test-Accuracy'); ax2.grid(True, axis='x')
ax2.axvline(max(test_accs), color='red', linestyle='--', alpha=0.5)

# LR vs. Test-Accuracy
ax3 = axes[1, 0]
lrs = [e['hyperparams']['lr'] for e in completed]
ax3.scatter(lrs, test_accs, s=100, color=colors[:len(completed)], zorder=5)
for i, (exp, lr, acc) in enumerate(zip(completed, lrs, test_accs)):
    ax3.annotate(exp['name'], (lr, acc), textcoords='offset points',
                  xytext=(5, 3), fontsize=7)
ax3.set_xscale('log')
ax3.set_title('LR vs. Test-Accuracy')
ax3.set_xlabel('Learning Rate (log)'); ax3.set_ylabel('Test-Accuracy'); ax3.grid(True)

# Trainingszeit
ax4 = axes[1, 1]
durations = [e['duration_s'] for e in completed]
ax4.bar(range(len(names)), durations, color=colors[:len(completed)], alpha=0.8)
ax4.set_xticks(range(len(names))); ax4.set_xticklabels(names, rotation=45, ha='right', fontsize=7)
ax4.set_title('Trainingszeit (Sekunden)')
ax4.set_ylabel('Sekunden'); ax4.grid(True, axis='y')

plt.tight_layout()
plt.savefig('F15_2_experiment_tracking.png', dpi=100)
plt.show()

print(f"\nExperiment-Log gespeichert: experiments.json")
print(f"Bestes Modell: {leaderboard[0]['name']} (Acc: {leaderboard[0]['final_metrics']['test_accuracy']:.4f})")


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 15: Projektarbeit & Best Practices
# Niveau: Fortgeschrittene
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(42)
tf.random.set_seed(42)

print("=== Fehleranalyse und Debugging ===\n")

# ── 1. Modell trainieren ──────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0
x_train_flat = x_train.reshape(-1, 784)
x_test_flat  = x_test.reshape(-1, 784)

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax')
], name='debug_model')
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Trainiere Modell...")
model.fit(x_train_flat, y_train, epochs=5, batch_size=256,
           validation_split=0.1, verbose=1)
_, acc = model.evaluate(x_test_flat, y_test, verbose=0)
print(f"Test-Accuracy: {acc:.4f}\n")

# ── 2. Vorhersagen berechnen ──────────────────────────────
probs      = model.predict(x_test_flat, verbose=0)
preds      = np.argmax(probs, axis=1)
losses_per = keras.losses.sparse_categorical_crossentropy(
    tf.constant(y_test, dtype=tf.int32),
    tf.constant(probs)
).numpy()

# ── 3. DEBUGGING SCHRITT 1: Hardest Examples ─────────────
print("=== DEBUG 1: Schwerste Beispiele (höchster Loss) ===")
sorted_by_loss = np.argsort(losses_per)[::-1]
top_k_hard = sorted_by_loss[:20]

print(f"Top-20 schwerste Beispiele:")
for idx in top_k_hard[:10]:
    print(f"  Sample {idx:5d}: Wahr={y_test[idx]}, Pred={preds[idx]}, "
          f"Konfidenz={probs[idx][preds[idx]]:.3f}, Loss={losses_per[idx]:.3f}")

# ── 4. DEBUGGING SCHRITT 2: Confusion Matrix ─────────────
print("\n=== DEBUG 2: Confusion Matrix ===")
cm = confusion_matrix(y_test, preds)
print(cm)

# Häufigste Verwechslungen
n_classes = 10
confusions = []
for i in range(n_classes):
    for j in range(n_classes):
        if i != j and cm[i, j] > 0:
            confusions.append({'true': i, 'pred': j, 'count': cm[i, j]})

confusions.sort(key=lambda x: x['count'], reverse=True)
print("\nTop-10 häufigste Verwechslungen:")
for c in confusions[:10]:
    print(f"  Wahr: {c['true']} → Pred: {c['pred']} | {c['count']} Fälle")

# ── 5. DEBUGGING SCHRITT 3: Per-Klassen Metriken ─────────
print("\n=== DEBUG 3: Per-Klassen Metriken ===")
print(classification_report(y_test, preds, digits=4))

per_class_acc      = cm.diagonal() / cm.sum(axis=1)
per_class_n_errors = cm.sum(axis=1) - cm.diagonal()

print("Klasse | Accuracy | Fehler | Häufigster Fehler (Pred → Count)")
print("-"*60)
for cls in range(n_classes):
    # Häufigster Fehler für diese Klasse
    row = cm[cls].copy()
    row[cls] = 0  # Korrekte nicht zählen
    most_confused_pred = np.argmax(row)
    most_confused_n    = row[most_confused_pred]
    print(f"  {cls}   | {per_class_acc[cls]:.4f}   | {per_class_n_errors[cls]:6d} | "
          f"→{most_confused_pred} ({most_confused_n})")

# ── 6. DEBUGGING SCHRITT 4: Systematische Fehler ─────────
print("\n=== DEBUG 4: Systematische Fehlermuster ===")

# Prüfe: Sind Fehler zufällig oder systematisch?
error_mask = preds != y_test
error_preds = preds[error_mask]
error_true  = y_test[error_mask]
error_conf  = probs[error_mask].max(axis=1)

print(f"Anzahl Fehler: {error_mask.sum()} / {len(y_test)} ({error_mask.mean()*100:.2f}%)")
print(f"Mittlere Konfidenz bei Fehlern: {error_conf.mean():.3f}")
print(f"Anteil hochkonfidenter Fehler (Konf>0.9): {(error_conf > 0.9).mean():.3f}")

# Welche Klassen werden am häufigsten falsch als was erkannt?
print("\nWelche Klassen werden oft füreinander gehalten?")
for c1, c2 in [(4, 9), (3, 5), (7, 1), (8, 3)]:
    n_c1_as_c2 = cm[c1, c2]
    n_c2_as_c1 = cm[c2, c1]
    print(f"  {c1}↔{c2}: {c1} als {c2}={n_c1_as_c2}, {c2} als {c1}={n_c2_as_c1}")

# ── 7. VISUALISIERUNG ─────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f'Fehleranalyse MNIST (Test-Acc: {acc:.4f})', fontsize=14)

# Confusion Matrix
ax = axes[0, 0]
im = ax.imshow(cm, cmap='Blues')
ax.set_title('Confusion Matrix')
ax.set_xlabel('Vorhergesagt'); ax.set_ylabel('Wahr')
ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
for i in range(n_classes):
    for j in range(n_classes):
        color = 'white' if cm[i,j] > cm.max()*0.5 else 'black'
        ax.text(j, i, str(cm[i,j]) if cm[i,j]>0 else '',
                 ha='center', va='center', fontsize=7, color=color)
plt.colorbar(im, ax=ax)

# Per-Klassen Accuracy
ax2 = axes[0, 1]
ax2.bar(range(n_classes), per_class_acc, color='steelblue', alpha=0.8)
ax2.axhline(acc, color='red', linestyle='--', label=f'Gesamt: {acc:.3f}')
ax2.set_title('Accuracy pro Klasse')
ax2.set_xlabel('Klasse'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(True, axis='y')

# Schwerste Beispiele
ax3 = axes[0, 2]
hard_imgs = x_test[top_k_hard[:9]]
grid = np.zeros((3*28, 3*28))
for i in range(3):
    for j in range(3):
        idx = i*3+j
        grid[i*28:(i+1)*28, j*28:(j+1)*28] = hard_imgs[idx]
ax3.imshow(grid, cmap='gray')
ax3.set_title('Schwerste 9 Beispiele (höchster Loss)')
ax3.axis('off')
# Beschriftungen
for i in range(3):
    for j in range(3):
        idx = i*3+j
        sidx = top_k_hard[idx]
        ax3.text(j*28+14, (i+1)*28-3, f'W:{y_test[sidx]}→P:{preds[sidx]}',
                  ha='center', color='red', fontsize=7)

# Loss-Verteilung: Korrekt vs. Falsch
ax4 = axes[1, 0]
correct_losses = losses_per[preds == y_test]
error_losses   = losses_per[preds != y_test]
ax4.hist(correct_losses, bins=50, alpha=0.7, color='blue',
          label=f'Korrekt (n={len(correct_losses)})', density=True)
ax4.hist(error_losses, bins=50, alpha=0.7, color='red',
          label=f'Falsch (n={len(error_losses)})', density=True)
ax4.set_title('Loss-Verteilung: Korrekt vs. Falsch')
ax4.set_xlabel('Loss'); ax4.set_ylabel('Dichte')
ax4.legend(); ax4.grid(True)

# Fehler-Konfidenz
ax5 = axes[1, 1]
ax5.hist(error_conf, bins=30, color='red', alpha=0.8, edgecolor='white')
ax5.axvline(0.9, color='orange', linestyle='--', label='0.9 Schwelle')
ax5.set_title('Konfidenz bei Fehler-Vorhersagen')
ax5.set_xlabel('Konfidenz'); ax5.set_ylabel('Anzahl')
ax5.legend(); ax5.grid(True)

# Verbesserungsvorschläge
ax6 = axes[1, 2]
ax6.axis('off')
suggestions = """Debuggung: Mögliche Verbesserungen

Beobachtete Probleme:
• Ziffer 4 wird oft als 9 verwechselt
• Ziffer 3 und 5 ähneln sich
• Hochkonfidente Fehler = Modell unsicher

Verbesserungsmaßnahmen:
1. Data Augmentation für schwierige
   Klassen (Rotation, Verschiebung)
2. Mehr Trainingsdaten für schwache
   Klassen
3. CNN statt Dense → Ortsunabhängig
4. Mixup-Augmentation für ähnliche
   Klassen
5. Confidence Calibration (Platt Scaling)
6. Fehleranalyse pro Klasse wiederholen"""
ax6.text(0.05, 0.95, suggestions, transform=ax6.transAxes,
          fontsize=9, va='top',
          bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout()
plt.savefig('F15_3_fehleranalyse.png', dpi=100)
plt.show()

print("\nFehleranalyse abgeschlossen!")
